In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

csv_path = Path.cwd() / "sikayet_var_yorum_link.csv"

if not csv_path.exists():
    print(f"CSV file not found at: {csv_path}")
    print(f"Current working directory: {Path.cwd()}")
    print("Available CSV files:")
    for csv_file in Path.cwd().glob("*.csv"):
        print(f"  - {csv_file}")
    if "csv_file" in globals() and csv_file.exists():
        csv_path = csv_file
    else:
        csv_candidates = list(Path.cwd().glob("*.csv"))
        if csv_candidates:
            csv_path = csv_candidates[0]
        else:
            raise FileNotFoundError(f"CSV bulunamadı: {csv_path}")

df = pd.read_csv(csv_path)
df.tail()

CSV file not found at: c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data\sikayet_var_yorum_link.csv
Current working directory: c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data
Available CSV files:
  - c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data\beko_urun_linki.csv
  - c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data\beko_urun_ozellik.csv
  - c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data\sikayet_var_urun_linki.csv
  - c:\Users\betul\Notes\YBST-veri-bilimi\ybst-veri-bilimi\beko\web-scraping-data\sikayet_var_yorum_linki.csv


,Urun_Adi_CSV,Urun_Linki,Urun_Adi_Sayfadan,Urun_Puan_Yuzdelik,Anahtar_Etiketler,Yorum_Linkleri,Sikayet_Sayisi
44,RHB 2910,https://www.sikayetvar.com/beko/rhb-2910,Beko\nRHB 2910,0.0,NaN,https://www.sikayetvar.com/beko/beko-blender-s...,2
45,HBS 5150 Floral,https://www.sikayetvar.com/beko/hbs-5150-floral,Beko\nHBS 5150 Floral,NaN,"el blender, yetkili servis, mağdur, numarası, ...",https://www.sikayetvar.com/beko/beko-cam-silme...,3
46,BKK 3099 HBS,https://www.sikayetvar.com/beko/bkk-3099-hbs,Beko\nBKK 3099 HBS,0.0,"aparat, blender yedek parça, yetkili servis, m...",https://www.sikayetvar.com/beko/beko-blender-y...,1
47,BKK 2153,https://www.sikayetvar.com/beko/bkk-2153,Beko\nBKK 2153,NaN,NaN,https://www.sikayetvar.com/beko/beko-blender-s...,1
48,BKK-1100 Steambox,https://www.sikayetvar.com/beko/bkk-1100-steambox,Beko\nBKK-1100 Steambox,NaN,NaN,https://www.sikayetvar.com/beko/buhar-makinesi...,13


In [3]:
missing_percentage = (df.isna().mean() * 100).sort_values(ascending=False)
missing_percentage

Anahtar_Etiketler     57.142857
Urun_Puan_Yuzdelik    48.979592
Yorum_Linkleri        14.285714
Urun_Adi_Sayfadan      0.000000
Urun_Linki             0.000000
Urun_Adi_CSV           0.000000
Sikayet_Sayisi         0.000000
dtype: float64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Urun_Adi_CSV        49 non-null     str    
 1   Urun_Linki          49 non-null     str    
 2   Urun_Adi_Sayfadan   49 non-null     str    
 3   Urun_Puan_Yuzdelik  25 non-null     float64
 4   Anahtar_Etiketler   21 non-null     str    
 5   Yorum_Linkleri      42 non-null     str    
 6   Sikayet_Sayisi      49 non-null     int64  
dtypes: float64(1), int64(1), str(5)
memory usage: 29.2 KB


In [5]:
import re

def count_links(s):
    if pd.isna(s) or not str(s).strip():
        return 0
    parts = re.split(r'[\s,;]+', str(s).strip())
    return sum(1 for p in parts if p.startswith('http'))

sample30 = df.sample(30, random_state=1).copy()
sample30['Yorum_Linkleri_Eleman_Sayisi'] = sample30['Yorum_Linkleri'].apply(count_links)
sample30[['Yorum_Linkleri_Eleman_Sayisi', 'Sikayet_Sayisi']]

,Yorum_Linkleri_Eleman_Sayisi,Sikayet_Sayisi
27,1,3
34,0,3
39,0,1
48,8,13
2,18,36
3,14,21
42,8,18
29,3,4
45,3,3
30,2,2


In [6]:
yorum_sayisi = pd.to_numeric(sample30["Yorum_Linkleri_Eleman_Sayisi"], errors="coerce")
sikayet_sayisi = pd.to_numeric(sample30["Sikayet_Sayisi"], errors="coerce")

greater_count = (yorum_sayisi > sikayet_sayisi).sum()
smaller_count = (yorum_sayisi < sikayet_sayisi).sum()

print(f"Büyük olduğu değer sayısı: {greater_count}")
print(f"Küçük olduğu değer sayısı: {smaller_count}")
smaller_count = (yorum_sayisi < sikayet_sayisi).sum()

print(f"Büyük olduğu değer sayısı: {greater_count}")
print(f"Küçük olduğu değer sayısı: {smaller_count}")

Büyük olduğu değer sayısı: 0
Küçük olduğu değer sayısı: 27
Büyük olduğu değer sayısı: 0
Küçük olduğu değer sayısı: 27


In [7]:
# Calculate the difference: positive if yorum_sayisi > sikayet_sayisi, negative if smaller
difference = yorum_sayisi - sikayet_sayisi
total_difference = difference.sum()

print(f"Toplam fark: {total_difference}")
print(f"\nDetaylı breakdown:")
print(f"Yorum sayısı > Şikayet sayısı: {greater_count} satır")
print(f"Yorum sayısı < Şikayet sayısı: {smaller_count} satır")
print(f"Yorum sayısı = Şikayet sayısı: {(yorum_sayisi == sikayet_sayisi).sum()} satır")
print(f"\nSatır satır farklar (+ = yorum fazla, - = şikayet fazla):\n{difference}")

# Aggregate difference by sign
positive_diff = difference[difference > 0].sum()
negative_diff = difference[difference < 0].sum()

print(f"\nToplam pozitif fark (+ işareti): {positive_diff}")
print(f"Toplam negatif fark (- işareti): {negative_diff}")
print(f"Net fark: {positive_diff + negative_diff}")

Toplam fark: -118

Detaylı breakdown:
Yorum sayısı > Şikayet sayısı: 0 satır
Yorum sayısı < Şikayet sayısı: 27 satır
Yorum sayısı = Şikayet sayısı: 3 satır

Satır satır farklar (+ = yorum fazla, - = şikayet fazla):
27    -2
34    -3
39    -1
48    -5
2    -18
3     -7
42   -10
29    -1
45     0
30     0
31    -1
38    -3
21    -5
35    -2
19    -9
41     0
36    -3
26    -1
22    -4
13    -3
40    -1
17   -10
44    -1
24    -1
23    -2
4    -12
32    -8
14    -2
10    -1
28    -2
dtype: int64

Toplam pozitif fark (+ işareti): 0
Toplam negatif fark (- işareti): -118
Net fark: -118
